# Contraction factors as identification ∘ collision — and only collision loses

**A-Mark9 / NEXUS · arena programme · §12 · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Self-contained: `itertools`, `numpy`, `collections` only. Continues §11 (subdivision as re-factorisation).

---

## 0. Where §11 left it

Three substrate/symmetry edits were measured one-sided exact, none losing a residual channel:

$$\begin{array}{lll}
\textbf{extension} & \mathcal T_X \subset \mathcal T_{X'} & \Delta_R \text{ onto, coker } 0\\
\textbf{hiding} & G \subset G' & R_{G'} \hookrightarrow R_G,\ \ker 0\\
\textbf{re-factorisation} & T \mapsto T_2 T_1 & F^*_R \text{ onto, coker } 0
\end{array}$$

and $\epsilon(C_N,3)=0,1,2,3$ for $N=6..9$ was reclassified as a change of visibility, not a creation of loops.
All three operations are inclusion- or refactor-friendly. Contraction is the first edit whose configuration
map is many-to-one, so it is the first that could genuinely destroy. The question was posed as:

> Does contraction create a nonzero kernel on the residual sector — and if so, is that kernel
> **equivalence** (identified representations) or **annihilation** (structure with no counterpart)?

## 1. Contraction is not one map

Contract the base edge $\{a,b\}$ by merging $b$ into $a$. On the $K$-token exclusion arena, a configuration
$S \ni a,b$ has **no image**: two occupancies would be forced onto one site. So contraction is only a partial
map on configurations, and it factors through the collision-free sub-arena

$$X' \xrightarrow{\ \text{delete } \{S : \{a,b\}\subseteq S\}\ } X'_0 \xrightarrow{\ q\ } X = X'/\{a,b\}.$$

**Collision.** $X'_0$ keeps configurations not holding both $a,b$, and transformations whose two endpoints
both survive. Dually, restriction $\rho: C^1(\mathcal A') \to C^1(\mathcal A'_0)$ commutes with $d_0$
and descends to residuals.

**Identification.** $q$ relabels vertices. On configurations it is $2\!:\!1$ on the fibres
$\{S,\ S-a+b\}$ and bijective elsewhere. On transformations, items along the collapsed edge $[a\to b]$ at $T$
are **degenerate** (both endpoints land on one configuration) and map to $0$; every other item maps to its
relabelled item with orientation sign. The claim, measured below, is

$$\partial\, q = q\, \partial \quad \text{on } C_*(\mathcal A'_0).$$

Dualising gives $q^*: C^1(\mathcal A) \to C^1(\mathcal A'_0)$ carrying exact to exact. The residual diagram is

$$R_X \xrightarrow{\ q^*_R\ } R_{X'_0} \xleftarrow{\ \rho_R\ } R_{X'}$$

**Common group.** The stabiliser of $\{a,b\}$ in $\operatorname{Aut}(X')$. It preserves $X'_0$ and descends
through the relabelling. Each descended element is checked to lie in $\operatorname{Aut}(X)$.

Only simple contractions are run ($\{a,b\}$ in no triangle), so $X$ is a simple graph.

## 2. Library

In [1]:
import itertools, numpy as np
from collections import defaultdict
ATOL = 1e-9
def rank_abs(M):
    if M.size == 0 or min(M.shape) == 0: return 0
    return int((np.linalg.svd(M, compute_uv=False) > ATOL).sum())
def colspace(M):
    if M.size == 0 or min(M.shape) == 0: return np.zeros((M.shape[0], 0))
    U, s, _ = np.linalg.svd(M, full_matrices=False); return U[:, s > ATOL]

def autos(E, n):
    Es = set(map(tuple, map(sorted, E))); deg = defaultdict(int); adj = defaultdict(set)
    for u, v in Es: deg[u] += 1; deg[v] += 1; adj[u].add(v); adj[v].add(u)
    out, p, used = [], [None]*n, [False]*n
    def bt(i):
        if i == n: out.append(tuple(p)); return
        for c in range(n):
            if used[c] or deg[c] != deg[i]: continue
            if all((j in adj[i]) == (p[j] in adj[c]) for j in range(i)):
                p[i] = c; used[c] = True; bt(i+1); used[c] = False
    bt(0); return out

class Arena:
    """K-token exclusion arena on graph (E, n); `allow` filters configurations (sub-arena)."""
    def __init__(s, E, n, K, allow=lambda S: True):
        s.n, s.K = n, K; s.E = sorted(set(map(tuple, map(sorted, E))))
        s.verts = [frozenset(S) for S in itertools.combinations(range(n), K) if allow(frozenset(S))]
        s.vi = {S: i for i, S in enumerate(s.verts)}
        s.items = []
        for e in s.E:
            for T in itertools.combinations([x for x in range(n) if x not in e], K-1):
                T = frozenset(T)
                if T | {e[0]} in s.vi and T | {e[1]} in s.vi: s.items.append((frozenset(e), T))
        s.idx = {it: i for i, it in enumerate(s.items)}
    def ends(s, it):
        e, T = it; u, w = sorted(e); return T | {u}, T | {w}        # oriented u -> w
    def d0(s):
        D = np.zeros((len(s.items), len(s.verts)))
        for it in s.items:
            A, B = s.ends(it); D[s.idx[it], s.vi[B]] += 1; D[s.idx[it], s.vi[A]] -= 1
        return D
    def components(s):
        return len(s.verts) - rank_abs(s.d0())
    def proj(s, G):
        P = np.zeros((len(s.items),)*2)
        for g in G:
            for it in s.items:
                e, T = it; u, w = sorted(e)
                j = (frozenset(g[i] for i in e), frozenset(g[i] for i in T))
                ju, jw = sorted(j[0]); sg = 1.0 if (g[u], g[w]) == (ju, jw) else -1.0
                P[s.idx[j], s.idx[it]] += sg/len(G)
        return P
    def residual(s, G):
        P = s.proj(G); Inv = colspace(P); Ex = colspace(P @ colspace(s.d0()))
        return colspace(Inv - Ex @ (Ex.T @ Inv)) if Ex.shape[1] else Inv

def contract(E, n, a, b):
    """Merge b into a (a<b); relabel. Refuses contractions that create multi-edges."""
    lab, k = {}, 0
    for v in range(n):
        if v == b: continue
        lab[v] = k; k += 1
    lab[b] = lab[a]
    Ec = set()
    for u, v in map(tuple, map(sorted, E)):
        if {u, v} == {a, b}: continue
        x, y = sorted((lab[u], lab[v]))
        if (x, y) in Ec: raise ValueError("multi-edge: edge lies in a triangle")
        Ec.add((x, y))
    return sorted(Ec), n-1, lab

## 3. The contraction machine

For each case the notebook builds $\mathcal A'$, $\mathcal A'_0$ and $\mathcal A$, the chain map $q$, and the
restriction $\rho$, then measures:

* the chain-map defect $\max|\partial q - q\partial|$;
* $\operatorname{rank} q^*_R$ and $\ker q^*_R$ (does **identification** lose?);
* $\operatorname{rank} \rho_R$ and $\ker \rho_R$ (does **collision** lose?);
* how $\operatorname{im} q^*$ and $\operatorname{im}\rho$ sit inside $R_{X'_0}$;
* the fate of the **full-symmetry** residual of $X'$: whether it survives deletion, and whether it comes from $X$.

Residual maps are computed as in §11. Each $R$ is the orthogonal complement of the invariant exact cochains
inside the invariant cochains, so the class of an invariant image vector is its orthogonal projection onto $R$.

In [2]:
def build(Ep, npr, K, e0):
    a, b = sorted(e0)
    E, n, lab = contract(Ep, npr, a, b)
    Ap = Arena(Ep, npr, K)
    A0 = Arena(Ep, npr, K, allow=lambda S: not {a, b} <= S)
    A  = Arena(E, n, K)
    # q : C_1(A0) -> C_1(A),  QV : C_0(A0) -> C_0(A)
    Q  = np.zeros((len(A.items), len(A0.items)))
    QV = np.zeros((len(A.verts), len(A0.verts)))
    for S, i in A0.vi.items(): QV[A.vi[frozenset(lab[x] for x in S)], i] = 1
    degen = 0
    for it in A0.items:
        e, T = it
        if set(e) == {a, b}: degen += 1; continue                      # degenerate -> 0
        As, Bs = A0.ends(it)
        qA, qB = frozenset(lab[x] for x in As), frozenset(lab[x] for x in Bs)
        j = (frozenset(lab[x] for x in e), frozenset(lab[x] for x in T)); jA, jB = A.ends(j)
        Q[A.idx[j], A0.idx[it]] += 1.0 if (jA, jB) == (qA, qB) else -1.0
    err = np.abs(A.d0().T @ Q - QV @ A0.d0().T).max()
    Rm = np.zeros((len(A0.items), len(Ap.items)))                      # restriction
    for it, i in A0.idx.items(): Rm[i, Ap.idx[it]] = 1
    Gp, GX = autos(Ep, npr), autos(E, n)
    stab = [g for g in Gp if {g[a], g[b]} == {a, b}]
    Gd = []
    for g in stab:
        h = [None]*n
        for v in range(npr): h[lab[v]] = lab[g[v]]
        Gd.append(tuple(h))
    assert all(h in set(GX) for h in Gd), "descended stabiliser not in Aut(X)"
    return dict(a=a, b=b, E=E, n=n, lab=lab, Ap=Ap, A0=A0, A=A, Q=Q, Rm=Rm, err=err,
                degen=degen, Gp=Gp, GX=GX, stab=stab, Gd=Gd)

def span_meet(U, V):
    if U.shape[1] == 0 or V.shape[1] == 0: return 0
    return U.shape[1] + V.shape[1] - rank_abs(np.hstack([U, V]))

def run(name, Ep, npr, K, e0):
    B = build(Ep, npr, K, e0)
    Ap, A0, A, Q, Rm = B['Ap'], B['A0'], B['A'], B['Q'], B['Rm']
    RX, R0, RP = A.residual(B['Gd']), A0.residual(B['stab']), Ap.residual(B['stab'])
    RXa, RPa   = A.residual(B['GX']), Ap.residual(B['Gp'])
    Mq, Mr = R0.T @ (Q.T @ RX), R0.T @ (Rm @ RP)
    rq, rr = rank_abs(Mq), rank_abs(Mr)
    Iq, Ir = colspace(R0 @ Mq), colspace(R0 @ Mr)
    Ia = colspace(R0 @ (R0.T @ (Rm @ RPa))) if RPa.shape[1] else np.zeros((R0.shape[0], 0))
    # trivial-group count of the collision kernel vs the Euler count of the deleted star
    I1 = [tuple(range(npr))]
    RP1, R01 = Ap.residual(I1), A0.residual(I1)
    ker1 = RP1.shape[1] - rank_abs(R01.T @ (Rm @ RP1))
    dI, dV = len(Ap.items) - len(A0.items), len(Ap.verts) - len(A0.verts)
    dc = A0.components() - Ap.components()
    out = dict(name=name, e0=tuple(sorted(e0)), E_p=len(Ap.E), E=len(A.E),
               V_p=len(Ap.verts), V_0=len(A0.verts), V=len(A.verts),
               I_p=len(Ap.items), I_0=len(A0.items), I=len(A.items), degen=B['degen'], err=B['err'],
               G_p=len(B['Gp']), G_X=len(B['GX']), G_c=len(B['stab']),
               RPa=RPa.shape[1], RP=RP.shape[1], R0=R0.shape[1], RX=RX.shape[1], RXa=RXa.shape[1],
               rq=rq, kq=RX.shape[1]-rq, rr=rr, kr=RP.shape[1]-rr,
               meet=span_meet(Iq, Ir), cover=rank_abs(np.hstack([Iq, Ir])),
               fa_surv=Ia.shape[1], fa_in_q=span_meet(Ia, Iq),
               ker1=ker1, dI=dI, dV=dV, dc=dc)
    print(f"\n### {name}   K={K}   contract {out['e0']}   |E'|={out['E_p']} -> |E|={out['E']}")
    print(f"    configs  X'={out['V_p']}  X'_0={out['V_0']}  X={out['V']}     "
          f"items  X'={out['I_p']}  X'_0={out['I_0']} (degenerate {out['degen']})  X={out['I']}")
    print(f"    chain map q on X'_0: max |dq - qd| = {out['err']:.1e}")
    print(f"    |Aut X'|={out['G_p']}  |Aut X|={out['G_X']}  |edge stabiliser|={out['G_c']}")
    print(f"    dim R:  X'@Aut={out['RPa']}  X'@stab={out['RP']}  X'_0@stab={out['R0']}  "
          f"X@stab={out['RX']}  X@Aut={out['RXa']}")
    print(f"    q*_R : R_X({out['RX']}) -> R_X'0({out['R0']})   rank={rq}  ker={out['kq']}  "
          f"coker={out['R0']-rq} (fibre distinctions identified)")
    print(f"    rho_R: R_X'({out['RP']}) -> R_X'0({out['R0']})  rank={rr}  ker={out['kr']} (collision loss)")
    print(f"    inside R_X'0:  im q* ∩ im rho = {out['meet']}   im q* + im rho = {out['cover']}")
    print(f"    trivial group: ker rho = {ker1};  removed items - removed configs - Δcomponents "
          f"= {dI} - {dV} - {dc} = {dI-dV-dc}")
    print(f"    full-Aut residual of X': {out['RPa']} channels, survive deletion {out['fa_surv']}, "
          f"inside im q* {out['fa_in_q']}")
    return out

## 4. Runs

Cycles (the $\epsilon(C_N,3)$ family, run backwards), a path, the subdivided cube (undoing §11 exactly), the cube, $K_{3,3}$ and Petersen. All at $K=3$.

In [3]:
C   = lambda m: [(i, (i+1) % m) for i in range(m)]
P   = lambda m: [(i, i+1) for i in range(m-1)]
Q3  = [(v, v ^ (1 << i)) for v in range(8) for i in range(3) if v < v ^ (1 << i)]
Q3s = [e for e in Q3 if e != (0, 1)] + [(0, 8), (1, 8)]          # §11 subdivision of Q3
K33 = [(i, j) for i in range(3) for j in range(3, 6)]
Pet = [(i, (i+1) % 5) for i in range(5)] + [(i, i+5) for i in range(5)] + [(5+i, 5+(i+2) % 5) for i in range(5)]

cases = [("C7 -> C6", C(7), 7, (0, 1)),
         ("C8 -> C7", C(8), 8, (0, 1)),
         ("C9 -> C8", C(9), 9, (0, 1)),
         ("P7 -> P6 (middle edge)", P(7), 7, (2, 3)),
         ("Q3' -> Q3 (undo subdivision)", Q3s, 9, (0, 8)),
         ("Q3 -> Q3/e", Q3, 8, (0, 1)),
         ("K33 -> K33/e", K33, 6, (0, 3)),
         ("Petersen -> Petersen/e", Pet, 10, (0, 1))]
results = [run(nm, E, n, 3, e0) for nm, E, n, e0 in cases]


### C7 -> C6   K=3   contract (0, 1)   |E'|=7 -> |E|=6
    configs  X'=35  X'_0=30  X=20     items  X'=70  X'_0=58 (degenerate 10)  X=36
    chain map q on X'_0: max |dq - qd| = 0.0e+00
    |Aut X'|=14  |Aut X|=12  |edge stabiliser|=2
    dim R:  X'@Aut=1  X'@stab=16  X'_0@stab=13  X@stab=7  X@Aut=0
    q*_R : R_X(7) -> R_X'0(13)   rank=7  ker=0  coker=6 (fibre distinctions identified)
    rho_R: R_X'(16) -> R_X'0(13)  rank=13  ker=3 (collision loss)
    inside R_X'0:  im q* ∩ im rho = 7   im q* + im rho = 13
    trivial group: ker rho = 7;  removed items - removed configs - Δcomponents = 12 - 5 - 0 = 7
    full-Aut residual of X': 1 channels, survive deletion 1, inside im q* 0

### C8 -> C7   K=3   contract (0, 1)   |E'|=8 -> |E|=7
    configs  X'=56  X'_0=50  X=35     items  X'=120  X'_0=105 (degenerate 15)  X=70
    chain map q on X'_0: max |dq - qd| = 0.0e+00
    |Aut X'|=16  |Aut X|=14  |edge stabiliser|=2
    dim R:  X'@Aut=2  X'@stab=30  X'_0@stab=26  X@stab=16  X@Aut=1
    q*_

## 5. Results

### 5.1 Structure and chain map

| contraction | $\partial q - q\partial$ | configs $X'$ / $X'_0$ / $X$ | items $X'$ / $X'_0$ (degen.) / $X$ | $|G_{X'}|,|G_X|,|G_\cap|$ |
|---|--:|---|---|---|
| $C_7\to C_6$ | $0.0$ | 35 / 30 / 20 | 70 / 58 (10) / 36 | 14, 12, 2 |
| $C_8\to C_7$ | $0.0$ | 56 / 50 / 35 | 120 / 105 (15) / 70 | 16, 14, 2 |
| $C_9\to C_8$ | $0.0$ | 84 / 77 / 56 | 189 / 171 (21) / 120 | 18, 16, 2 |
| $P_7\to P_6$ | $0.0$ | 35 / 30 / 20 | 60 / 49 (10) / 30 | 2, 2, 1 |
| $Q_3'\to Q_3$ | $0.0$ | 84 / 77 / 56 | 273 / 246 (21) / 180 | 4, 48, 2 |
| $Q_3\to Q_3/e$ | $0.0$ | 56 / 50 / 35 | 180 / 153 (15) / 110 | 48, 4, 4 |
| $K_{3,3}\to K_{3,3}/e$ | $0.0$ | 20 / 16 / 10 | 54 / 38 (6) / 24 | 72, 8, 8 |
| Petersen $\to$ Pet$/e$ | $0.0$ | 120 / 112 / 84 | 420 / 382 (28) / 294 | 120, 8, 8 |

$$\boxed{\ q \text{ is an exact chain map on the collision-free arena in all eight cases}\ }$$

### 5.2 Residual maps (common group = edge stabiliser)

| contraction | $\dim R$: $X'$@Aut / $X'$@$\cap$ / $X'_0$@$\cap$ / $X$@$\cap$ / $X$@Aut | $\operatorname{rank} q^*_R$ | **$\ker q^*_R$** | coker $q^*_R$ | $\operatorname{rank}\rho_R$ | **$\ker\rho_R$** | $\operatorname{im}q^*\subseteq\operatorname{im}\rho$ |
|---|---|--:|--:|--:|--:|--:|:-:|
| $C_7\to C_6$ | 1 / 16 / 13 / 7 / 0 | 7 | **0** | 6 | 13 | **3** | ✓ |
| $C_8\to C_7$ | 2 / 30 / 26 / 16 / 1 | 16 | **0** | 10 | 26 | **4** | ✓ |
| $C_9\to C_8$ | 3 / 50 / 45 / 30 / 2 | 30 | **0** | 15 | 45 | **5** | ✓ |
| $P_7\to P_6$ | 12 / 26 / 20 / 11 / 5 | 11 | **0** | 9 | 20 | **6** | ✓ |
| $Q_3'\to Q_3$ | 42 / 93 / 83 / 60 / 1 | 60 | **0** | 23 | 83 | **10** | ✓ |
| $Q_3\to Q_3/e$ | 1 / 26 / 22 / 15 / 15 | 15 | **0** | 7 | 22 | **4** | ✓ |
| $K_{3,3}\to K_{3,3}/e$ | 0 / 3 / 2 / 1 / 1 | 1 | **0** | 1 | 2 | **1** | ✓ |
| Petersen $\to$ Pet$/e$ | 1 / 33 / 30 / 22 / 22 | 22 | **0** | 8 | 30 | **3** | ✓ |

$$\boxed{\ \text{identification: } \ker q^*_R = 0 \qquad \text{collision: } \ker\rho_R > 0 \quad \text{in every case}\ }$$

The cycle cokernels $6, 10, 15$ equal the §11 counts of re-factored transformations $6, 10, 15$, which are
$\binom{N-3}{2}$ for $C_N \to C_{N-1}$. The collision kernels on cycles are $N-4$ at $\mathbb Z_2$.

### 5.3 Identification is safe — proof, not measurement

**Claim.** $q^*: C^1(\mathcal A)/B^1 \to C^1(\mathcal A'_0)/B^1$ is injective.

**Proof.**

1. Suppose $q^*f = d_0 g'$ on $\mathcal A'_0$.
2. For each fibre $\{S, S-a+b\}$ in $\mathcal A'_0$, the degenerate item $[a\to b]$ at $T=S-a$ lies in $\mathcal A'_0$, since $T$ holds neither $a$ nor $b$. It connects the two points of the fibre.
3. $q^*f$ vanishes on that item, so $g'(S)=g'(S-a+b)$.
4. Hence $g'$ is constant on fibres and descends to $g$ on configurations of $\mathcal A$.
5. Every item of $\mathcal A$ has a non-degenerate preimage in $\mathcal A'_0$. For an item meeting the merged vertex, take the preimage edge at whichever of $a,b$ it uses, with $T$ lifted away from the collision.
6. So $q$ is onto items, and $f = d_0 g$.

With a group, $q$ is equivariant for the descended stabiliser and the orthogonal projectors commute with
$q^*$, so the argument passes to invariant residuals. $\square$

Merging configurations kills nothing. Its cokernel, the fibre distinctions carried by $X'_0$ but not by $X$,
is **identified**, not annihilated.

### 5.4 Collision loses — and exactly by the Euler count

In [4]:
print(f"{'contraction':34s} {'ker rho (trivial G)':>20s} {'removed items':>14s} {'removed configs':>16s} {'Δcomp':>6s} {'Euler':>6s}")
for r in results:
    print(f"{r['name']:34s} {r['ker1']:>20d} {r['dI']:>14d} {r['dV']:>16d} {r['dc']:>6d} {r['dI']-r['dV']-r['dc']:>6d}")

contraction                         ker rho (trivial G)  removed items  removed configs  Δcomp  Euler
C7 -> C6                                              7             12                5      0      7
C8 -> C7                                              9             15                6      0      9
C9 -> C8                                             11             18                7      0     11
P7 -> P6 (middle edge)                                6             11                5      0      6
Q3' -> Q3 (undo subdivision)                         20             27                7      0     20
Q3 -> Q3/e                                           21             27                6      0     21
K33 -> K33/e                                         12             16                4      0     12
Petersen -> Petersen/e                               30             38                8      0     30


$\rho$ is onto on residuals automatically, because restriction is onto cochains and exact classes restrict to exact
classes. The kernel dimension is therefore forced by counting:

$$\dim\ker\rho \;=\; \big(|I'| - |V'| + c'\big) - \big(|I'_0| - |V'_0| + c'_0\big) \;=\; \#\text{removed items} - \#\text{removed configs} - \Delta c.$$

Components never change in these runs ($\Delta c = 0$). The measured trivial-group kernel matches the formula in
every case: $7, 9, 11, 6, 20, 21, 12, 30$ (see the table above).

These classes are loops threading through the **excluded states**, configurations holding both merged sites.
They are not symmetry redundancy, because the group is the same on both sides of $\rho$. They are not identified
representations either, because they have no image to be identified with.

$$\boxed{\ \text{genuine structural loss is located at exclusion, not at identification}\ }$$

### 5.5 Cross-check with §11: where the full-symmetry channels go

In [5]:
hdr = ("contraction", "dim R_X'@Aut", "survive deletion", "inside im q*")
print(f"{hdr[0]:34s} {hdr[1]:>15s} {hdr[2]:>17s} {hdr[3]:>13s}")
for r in results:
    print(f"{r['name']:34s} {r['RPa']:>15d} {r['fa_surv']:>17d} {r['fa_in_q']:>13d}")

contraction                           dim R_X'@Aut  survive deletion  inside im q*
C7 -> C6                                         1                 1             0
C8 -> C7                                         2                 2             0
C9 -> C8                                         3                 3             0
P7 -> P6 (middle edge)                          12                12             3
Q3' -> Q3 (undo subdivision)                    42                42            19
Q3 -> Q3/e                                       1                 1             0
K33 -> K33/e                                     0                 0             0
Petersen -> Petersen/e                           1                 1             0


| contraction | full-Aut channels of $X'$ | survive collision deletion | inside $\operatorname{im} q^*$ |
|---|--:|--:|--:|
| $C_7\to C_6$ | 1 | **1** | **0** |
| $C_8\to C_7$ | 2 | **2** | **0** |
| $C_9\to C_8$ | 3 | **3** | **0** |
| $Q_3\to Q_3/e$ | 1 | **1** | **0** |
| Petersen $\to$ Pet$/e$ | 1 | **1** | **0** |
| $P_7\to P_6$ | 12 | 12 | 3 |
| $Q_3'\to Q_3$ | 42 | 42 | 19 |
| $K_{3,3}\to K_{3,3}/e$ | 0 | – | – |

In every case the full-symmetry channels **never fall into the collision kernel**. Where the full group is large
(cycles, cube, Petersen) they lie **entirely outside** the pullback from $X$. Where it is small ($P_7$, $Q_3'$ with
$|\operatorname{Aut}|\le 4$) they split, with 3 of 12 and 19 of 42 coming from $X$. This mirrors the §11 split
between the cycle family and $P_6$, $Q_3$. For cycles they are exactly the $\epsilon(C_N,3)$ channels that §11 showed pull back injectively under subdivision.
The same cochains are therefore read two ways:

* **subdivision** $F^*$ sees them as cochains already present on the shorter arena;
* **contraction** $q^*$ sees them as fibre distinctions, which the merge identifies.

So subdivision and contraction are **not adjoint on residuals**. The identity $F^*q^* = \mathrm{id}$ holds when the
contracted edge is itself a subdivision edge, the $Q_3'\to Q_3$ row, where $q\circ F = \mathrm{id}$ on chains.
There $R_X$ is a retract of $R_{X'_0}$, and nothing stronger follows.

---

## 6. The five mechanisms

$$\begin{array}{llll}
\textbf{extension} & \mathcal T_X \subset \mathcal T_{X'} & \Delta_R \text{ onto} & \text{coker } 0 \ \text{(measured)}\\
\textbf{hiding} & G \subset G' & R_{G'} \hookrightarrow R_G & \ker 0 \ \text{(measured)}\\
\textbf{re-factorisation} & T \mapsto T_2T_1 & F^*_R \text{ onto} & \text{coker } 0 \ \text{(measured)}\\
\textbf{identification} & S \sim S-a+b & q^*_R \hookrightarrow & \ker 0 \ \text{(proved)}\\
\textbf{collision} & \{a,b\}\subseteq S \text{ excluded} & \rho_R \text{ onto} & \ker = \chi(\text{excluded star}) > 0
\end{array}$$

$$\boxed{\ \text{edits extend or re-factor; symmetry hides; merging identifies; only exclusion erases}\ }$$

This refines the §11 statement. The operation expected to destroy (contraction) splits into a safe half and a
lossy half. The lossy half is not the many-to-one merge. It is the prohibition on two occupancies sharing one
location.

## 7. Relation to the constraint chain

The chain from constraint to reality reads:

* **Law 2.** No matter may hold a location without the potential to be moved from it.
* **New potential.** Collision.
* **Constraint 3.** All change is equal.
* **Law 3.** Open.

On the arena, every elementary edit except one preserves residual structure up to factorisation, visibility or
identification. The exception is collision under exclusion, and its cost is a fixed count, not a free parameter.

**Law 3 (candidate, conjecture — not established).** Every transformation of substrate preserves continuation up
to re-factorisation, visibility and identification. Structure is lost only where two occupancies are forced into
one location, and the loss equals the Euler count of the excluded states.

Read this way, "all change is equal" is not a conserved scalar. It is the statement that the non-colliding edits
are all one-sided exact in the safe direction. The price appears only at exclusion.

## 8. Not established

* Eight contractions, $K=3$ only, simple contractions only. Edges in triangles create multi-edges and were not run.
* The kernel formula for $\rho$ is an Euler identity once $\rho$ is onto. Its content is the **location** of the
  loss, not its size.
* That full-symmetry channels never enter $\ker\rho$ is measured in seven cases (one is empty) and not proved. When they lie wholly outside $\operatorname{im}q^*$ is not characterised.
* The equivariant version of §5.3 is sketched, not written in full.
* Law 3 above is a reading of the measurements, not a theorem. Nothing here is about mass, memory or matter;
  $R$ is a cochain sector.

## 9. Open

1. **Capacity-2 merged site.** Relax exclusion at the merged vertex, so the collision configurations get an image.
   If $\ker\rho$ vanishes, the loss is attributable to exclusion alone. This is the decisive next test.
2. Prove that the full-symmetry residual avoids $\ker\rho$, or find a counterexample (triangles, larger $K$).
3. Contractions through triangles (multigraph arenas): does parallel-edge creation add a third factor?
4. Iterated edits. The composite of subdivision then contraction is the identity on the base graph but not on
   residual bookkeeping. Measure the loop $X \to X' \to X$ and ask whether its defect is exactly the collision count.